# Baseline Survival Models Comparison: Age/Sex vs Epigenetic Clocks
This notebook fits and evaluates two baseline survival models (Base-A and Base-B) that use no methylation features:
1. **Base-A:** Chronological Age + Sex
2. **Base-B:** Chronological Age + Sex + 4 clinical markers (Total Cholesterol, HDL, Glucose, Triglycerides)

We compare these baseline models against:
* **CpG-only:** The Coxnet model trained solely on 10 epigenetic clock CpG sites.
* **CpG+clinical:** Our final Coxnet model trained on 10 CpG sites and the 4 clinical markers.

We evaluate performance using Harrell's Concordance Index (C-index), bootstrap 95% confidence intervals, and paired bootstrap differences ($\Delta C = C(\text{CpG+clinical}) - C(\text{Base-B})$).


In [ ]:
import sys
from pathlib import Path

# Resolve repository root and import config
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent:
    if (_repo_root / "config.py").exists():
        if str(_repo_root) not in sys.path:
            sys.path.insert(0, str(_repo_root))
        break
    _repo_root = _repo_root.parent

import config

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored

# Configure Styling
sns.set_theme(style="white", context="talk")
plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.titlesize": 18
})

# Setup directories (using relative path to base folder)
BASE_DIR = config.PROJECT_ROOT
DATA_DIR = config.DATA_DIR
PRED_DIR = config.PREDICTIONS_DIR
PLOT_DIR = BASE_DIR / "plots" / "combined"

PLOT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)

print("Environment set up successfully. Base directory:", BASE_DIR)

In [ ]:
# Load raw datasets
fhs_data = pd.read_csv(DATA_DIR / "FHS_non_normalized.csv", low_memory=False)
fhs_data = fhs_data.rename(columns={"Share_ID": "patient_id"})

whi_data = pd.read_csv(DATA_DIR / "WHI_non_normalized.csv", low_memory=False)
whi_data = whi_data.rename(columns={"SUBJECT_ID": "patient_id"})

# Identify features
DROP_COLS = ["patient_id", "SUBJECT_ID", "Share_ID", "dbGaP_Subject_ID", "time", "event", "cohort", "batch", "age", "sex"]
fhs_features = [c for c in fhs_data.columns if c not in DROP_COLS]
whi_features = [c for c in whi_data.columns if c not in DROP_COLS]
common_features = sorted(list(set(fhs_features).intersection(set(whi_features))))

FHS = fhs_data[["patient_id", "time", "event", "age", "sex"] + common_features].copy()
WHI = whi_data[["patient_id", "time", "event", "age", "sex"] + common_features].copy()

# Stratified Splitting for FHS (Train 60%, Val 20%, Test 20%)
FHS["event_stratify"] = FHS["event"].astype(int)

train_val_df, test_df = train_test_split(
    FHS, test_size=0.20, random_state=SEED, stratify=FHS["event_stratify"]
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.25, random_state=SEED, stratify=train_val_df["event_stratify"]
)

train_df = train_df.drop(columns=["event_stratify"]).reset_index(drop=True)
val_df = val_df.drop(columns=["event_stratify"]).reset_index(drop=True)
test_df = test_df.drop(columns=["event_stratify"]).reset_index(drop=True)
FHS = FHS.drop(columns=["event_stratify"])

print(f"Loaded splits:")
print(f"  FHS Train : {len(train_df)} rows")
print(f"  FHS Val   : {len(val_df)} rows")
print(f"  FHS Test  : {len(test_df)} rows")
print(f"  WHI Ext   : {len(WHI)} rows")

In [ ]:
# Load existing CpG predictions
train_preds_cpg_clin = pd.read_csv(PRED_DIR / "coxnet_train_V3_OPTUNA_SS.csv")
val_preds_cpg_clin = pd.read_csv(PRED_DIR / "coxnet_val_V3_OPTUNA_SS.csv")
test_preds_cpg_clin = pd.read_csv(PRED_DIR / "coxnet_test_V3_OPTUNA_SS.csv")
whi_preds_cpg_clin = pd.read_csv(PRED_DIR / "coxnet_whi_V3_OPTUNA_SS.csv")

train_preds_cpg_only = pd.read_csv(PRED_DIR / "coxnet_train_V3_OPTUNA_SS_CPG.csv")
val_preds_cpg_only = pd.read_csv(PRED_DIR / "coxnet_val_V3_OPTUNA_SS_CPG.csv")
test_preds_cpg_only = pd.read_csv(PRED_DIR / "coxnet_test_V3_OPTUNA_SS_CPG.csv")
whi_preds_cpg_only = pd.read_csv(PRED_DIR / "coxnet_whi_V3_OPTUNA_SS_CPG.csv")

# Helper to merge data with predictions and check alignment
def merge_and_align(df, preds_clin, preds_cpg):
    df = df.copy()
    df['patient_id'] = df['patient_id'].astype(int)
    preds_clin = preds_clin.copy()
    preds_clin['patient_id'] = preds_clin['patient_id'].astype(int)
    preds_cpg = preds_cpg.copy()
    preds_cpg['patient_id'] = preds_cpg['patient_id'].astype(int)
    
    m = pd.merge(df, preds_clin[['patient_id', 'risk_score']], on='patient_id')
    m = m.rename(columns={'risk_score': 'risk_cpg_clin'})
    
    m = pd.merge(m, preds_cpg[['patient_id', 'risk_score']], on='patient_id')
    m = m.rename(columns={'risk_score': 'risk_cpg_only'})
    
    assert len(m) == len(df), f"Mismatch in length: raw={len(df)}, merged={len(m)}"
    return m
    
train_df = merge_and_align(train_df, train_preds_cpg_clin, train_preds_cpg_only)
val_df = merge_and_align(val_df, val_preds_cpg_clin, val_preds_cpg_only)
test_df = merge_and_align(test_df, test_preds_cpg_clin, test_preds_cpg_only)
WHI_df = merge_and_align(WHI, whi_preds_cpg_clin, whi_preds_cpg_only)

In [ ]:
# Preprocessing: standardize continuous covariates with training-split means/SDs
continuous_A = ["age"]
continuous_B = ["age", "Glucose", "HDL", "Total_Cholesterol", "Triglycerides"]

scaler_A = StandardScaler()
scaler_A.fit(train_df[continuous_A])

scaler_B = StandardScaler()
scaler_B.fit(train_df[continuous_B])

def apply_scaling(df, scaler, continuous_cols):
    df_scaled = df.copy()
    df_scaled[continuous_cols] = scaler.transform(df[continuous_cols])
    return df_scaled
    
train_df_A = apply_scaling(train_df, scaler_A, continuous_A)
val_df_A = apply_scaling(val_df, scaler_A, continuous_A)
test_df_A = apply_scaling(test_df, scaler_A, continuous_A)
WHI_df_A = apply_scaling(WHI_df, scaler_A, continuous_A)

train_df_B = apply_scaling(train_df, scaler_B, continuous_B)
val_df_B = apply_scaling(val_df, scaler_B, continuous_B)
test_df_B = apply_scaling(test_df, scaler_B, continuous_B)
WHI_df_B = apply_scaling(WHI_df, scaler_B, continuous_B)

In [ ]:
# Fit standard Cox PH models on training split
def to_structured_array(y_df):
    return np.array(
        list(zip(y_df["event"], y_df["time"])),
        dtype=[("event", "?"), ("time", "<f8")]
    )

y_train_surv = to_structured_array(train_df[['event', 'time']])

# Fit Base-A: Age + Sex
base_A_model = CoxPHSurvivalAnalysis()
base_A_model.fit(train_df_A[['age', 'sex']], y_train_surv)

# Fit Base-B: Age + Sex + Clinical Markers
base_B_model = CoxPHSurvivalAnalysis()
base_B_model.fit(train_df_B[['age', 'sex', 'Glucose', 'HDL', 'Total_Cholesterol', 'Triglycerides']], y_train_surv)

# Save predictions for later comparison
splits = {
    'Train': (train_df, train_df_A, train_df_B),
    'Val': (val_df, val_df_A, val_df_B),
    'Test': (test_df, test_df_A, test_df_B),
    'WHI': (WHI_df, WHI_df_A, WHI_df_B)
}

risk_scores = {}
for name, (orig_df, df_A, df_B) in splits.items():
    risk_scores[name] = {
        'Base-A': base_A_model.predict(df_A[['age', 'sex']]),
        'Base-B': base_B_model.predict(df_B[['age', 'sex', 'Glucose', 'HDL', 'Total_Cholesterol', 'Triglycerides']]),
        'CpG-only': orig_df['risk_cpg_only'].values,
        'CpG+clinical': orig_df['risk_cpg_clin'].values,
        'event': orig_df['event'].values.astype(bool),
        'time': orig_df['time'].values
    }
print("Models fitted and predictions stored.")

In [ ]:
# Compute C-index and bootstrap CIs (1000 resamples)
n_bootstraps = 1000
rng = np.random.RandomState(SEED)

models = ['Base-A', 'Base-B', 'CpG-only', 'CpG+clinical']
cindex_results = {model: {} for model in models}

for split_name, data in risk_scores.items():
    event = data['event']
    time = data['time']
    n_samples = len(event)
    
    # Draw bootstrap indices for this split
    boot_indices = rng.choice(n_samples, size=(n_bootstraps, n_samples), replace=True)
    
    for model in models:
        risk = data[model]
        c_orig = concordance_index_censored(event, time, risk)[0]
        
        boot_cis = []
        for b in range(n_bootstraps):
            idx = boot_indices[b]
            ev_b = event[idx]
            ti_b = time[idx]
            rk_b = risk[idx]
            try:
                ci = concordance_index_censored(ev_b, ti_b, rk_b)[0]
                boot_cis.append(ci)
            except Exception:
                pass
        boot_cis = np.array(boot_cis)
        ci_lower = np.percentile(boot_cis, 2.5)
        ci_upper = np.percentile(boot_cis, 97.5)
        
        cindex_results[model][split_name] = {
            'original': c_orig,
            'ci_lower': ci_lower,
            'ci_upper': ci_upper,
            'boot_runs': boot_cis
        }

In [ ]:
# Compute paired differences DeltaC = C(CpG+clinical) - C(Base-B) on Test and WHI
delta_results = {}

for split_name in ['Test', 'WHI']:
    data = risk_scores[split_name]
    event = data['event']
    time = data['time']
    risk_cpg = data['CpG+clinical']
    risk_base = data['Base-B']
    n_samples = len(event)
    
    c_cpg_orig = concordance_index_censored(event, time, risk_cpg)[0]
    c_base_orig = concordance_index_censored(event, time, risk_base)[0]
    delta_observed = c_cpg_orig - c_base_orig
    
    rng_delta = np.random.RandomState(SEED)
    boot_indices_delta = rng_delta.choice(n_samples, size=(n_bootstraps, n_samples), replace=True)
    
    delta_boots = []
    for b in range(n_bootstraps):
        idx = boot_indices_delta[b]
        ev_b = event[idx]
        ti_b = time[idx]
        rk_cpg_b = risk_cpg[idx]
        rk_base_b = risk_base[idx]
        try:
            ci_cpg = concordance_index_censored(ev_b, ti_b, rk_cpg_b)[0]
            ci_base = concordance_index_censored(ev_b, ti_b, rk_base_b)[0]
            delta_boots.append(ci_cpg - ci_base)
        except Exception:
            pass
            
    delta_boots = np.array(delta_boots)
    mean_delta = np.mean(delta_boots)
    ci_lower = np.percentile(delta_boots, 2.5)
    ci_upper = np.percentile(delta_boots, 97.5)
    
    # Two-sided bootstrap p-value
    null_dist = delta_boots - mean_delta
    p_val = np.mean(np.abs(null_dist) >= np.abs(delta_observed))
    
    delta_results[split_name] = {
        'observed': delta_observed,
        'mean_delta': mean_delta,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'p_value': p_val,
        'boot_runs': delta_boots
    }

In [ ]:
# Plot 1: Forest Plot of Concordance Indices across Cohorts
fig, ax = plt.subplots(figsize=(12, 8))

split_names = ['Train', 'Val', 'Test', 'WHI']
model_names = ['Base-A', 'Base-B', 'CpG-only', 'CpG+clinical']
model_colors = {
    'Base-A': '#3F51B5',      # Indigo
    'Base-B': '#009688',      # Teal
    'CpG-only': '#FF9800',    # Orange
    'CpG+clinical': '#9C27B0'  # Purple
}
model_labels = {
    'Base-A': 'Base-A (Age + Sex)',
    'Base-B': 'Base-B (Age + Sex + Clinical)',
    'CpG-only': 'CpG-only Model',
    'CpG+clinical': 'CpG+clinical Model'
}

y_pos_ticks = []
y_labels = []

# Position spacing logic
y_offset = 0.0
for split in split_names:
    y_pos_ticks.append(y_offset + 1.5 * 0.15)
    y_labels.append(f"{split} split")
    
    for i, model in enumerate(model_names):
        res = cindex_results[model][split]
        c_val = res['original']
        l_val = res['ci_lower']
        u_val = res['ci_upper']
        
        y_pos = y_offset + i * 0.15
        
        xerr = np.array([[c_val - l_val], [u_val - c_val]])
        ax.errorbar(
            x=c_val,
            y=y_pos,
            xerr=xerr,
            fmt='s',
            color=model_colors[model],
            ecolor='#37474F',
            elinewidth=2,
            capsize=4,
            markersize=8,
            label=model_labels[model] if split == 'Train' else ""
        )
    y_offset += 1.0

ax.set_yticks(y_pos_ticks)
ax.set_yticklabels(y_labels, fontweight='bold')
ax.invert_yaxis()
ax.set_xlabel("Harrell's Concordance Index (C-index)")
ax.set_title("Survival Discrimination C-index Forest Plot", fontweight='bold', pad=15)
ax.legend(loc='lower left', bbox_to_anchor=(0.0, 1.02), ncol=2, frameon=True, facecolor='white', edgecolor='none')
ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plot_path = PLOT_DIR / "baseline_comparison_forest.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved forest plot to: {plot_path}")
plt.show()

In [ ]:
# Plot 2: Paired Bootstrap Differences DeltaC Distribution (Test and WHI)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, split_name in enumerate(['Test', 'WHI']):
    ax = axes[idx]
    res = delta_results[split_name]
    boots = res['boot_runs']
    obs = res['observed']
    
    sns.histplot(boots, kde=True, color='skyblue', ax=ax, edgecolor='white')
    ax.axvline(x=0, color='gray', linestyle='--', linewidth=1.5, label='Null Hypothesis (DeltaC = 0)')
    ax.axvline(x=obs, color='crimson', linestyle='-', linewidth=2.0, label=f'Observed DeltaC = {obs:.4f}')
    
    p_str = f"p < 0.0001" if res['p_value'] < 0.0001 else f"p = {res['p_value']:.4f}"
    ax.set_title(f"Bootstrap Distribution of DeltaC on {split_name}\n(95% CI: {res['ci_lower']:.4f} to {res['ci_upper']:.4f}, {p_str})", fontsize=13)
    ax.set_xlabel("Delta C-index (CpG+clinical - Base-B)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=9, loc='upper left')

plt.suptitle("Paired Bootstrap Differences Distribution", fontweight='bold', y=1.02)
plt.tight_layout()
plot_path = PLOT_DIR / "baseline_comparison_deltas.png"
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved delta distribution plot to: {plot_path}")
plt.show()

In [ ]:
# Concordance Index Markdown Table
columns = ['Train', 'Val', 'Test', 'WHI']
rows = []
for model in models:
    row_data = [model]
    for col in columns:
        res = cindex_results[model][col]
        row_data.append(f"{res['original']:.4f} ({res['ci_lower']:.4f} - {res['ci_upper']:.4f})")
    rows.append(row_data)
    
df_table = pd.DataFrame(rows, columns=['Model'] + columns)
print("=== Concordance Index (95% CI) ===")
print(df_table.to_markdown(index=False))

print("\n=== Paired Differences ===")
delta_rows = []
for split_name in ['Test', 'WHI']:
    res = delta_results[split_name]
    p_str = f"{res['p_value']:.4e}" if res['p_value'] < 0.0001 else f"{res['p_value']:.4f}"
    delta_rows.append([
        f"DeltaC (CpG+clinical - Base-B) on {split_name}",
        f"{res['observed']:.4f} ({res['ci_lower']:.4f} - {res['ci_upper']:.4f})",
        p_str
    ])
df_delta_table = pd.DataFrame(delta_rows, columns=['Metric', 'Estimate (95% CI)', 'Bootstrap p-value'])
print(df_delta_table.to_markdown(index=False))

# Print explicit requested C-index
print(f"\nExplicit Base-B Test split C-index: {cindex_results['Base-B']['Test']['original']:.4f}")